# A100 OCR 캐시 생성 — 배치 크기 자동 비교

기존 Colab 노트북을 **정상 중지하여 Drive 저장을 마친 뒤** 이 파일을 실행하세요.
같은 캐시에 두 노트북을 동시에 실행하지 마세요. 기존 Spotting 캐시를 이어서 처리합니다.

1. A100 런타임에서 설치 셀 실행 → 필요하면 세션 재시작 → 다음 셀부터 실행.
2. `ZIP_PATH`, `DRIVE_BASE`를 자신의 Drive 경로에 맞춥니다.
3. 실행 셀은 남은 데이터 중 고정된 최대 16장으로 배치 1/2/4/8을 비교합니다.
4. 단일 처리와 샘플 출력이 모두 같은 후보 중 가장 빠른 크기로 전체를 처리합니다.

측정에는 원래 1024토큰 한도를 사용합니다. 벤치마크는 몇 분 걸릴 수 있습니다.
샘플 일치는 전체 결과 동일성이나 정확도 보장이 아닙니다. GPU의 수치 연산 차이로
배치 결과가 달라지는 후보는 보수적으로 제외합니다. 속도 개선이 없으면 배치 1을 씁니다.
해상도, 프롬프트, 토큰 한도는 원래 설정을 유지합니다.

공식 배치 처리 안내: https://huggingface.co/docs/transformers/en/model_doc/paddleocr_vl#batched-inference

## 1. 패키지 설치

PaddleOCR-VL-1.6은 **transformers 5.x**가 필요합니다. Colab 기본은 4.x라 반드시 설치해야 합니다.

**설치 후 반드시 `런타임 - 세션 다시 시작`을 하고, 이 셀은 건너뛴 채 2번부터 다시 실행하세요.**

In [ ]:
!pip -q install "transformers>=5.3.0,<6" accelerate
!pip -q install pillow pandas tqdm

In [ ]:
import transformers, torch
print("transformers", transformers.__version__, "/ torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert transformers.__version__ >= "5", "세션을 다시 시작한 뒤 2번 셀부터 실행하세요."

## 2. Drive 마운트 · 데이터 준비

Drive에는 두 가지를 둡니다.
- **입력**: `ssafy-16-2-ai.zip` (대회 데이터)
- **출력**: OCR 캐시. 런타임이 끊겨도 살아남도록 Drive에 씁니다

압축 해제는 몇 분 걸립니다. 이미 `/content/data`가 있으면 건너뜁니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"

def local_csv_files(name):
    for parent, dirs, files in os.walk('/content'):
        dirs[:] = [d for d in dirs if d not in {'drive', '.cache', 'sample_data'}]
        if name in files:
            yield Path(parent) / name

assert os.path.isfile(ZIP_PATH), f"zip을 찾지 못했습니다: {ZIP_PATH}"
if list(local_csv_files('train.csv')):
    print('로컬 데이터가 있어 압축 해제를 건너뜁니다.')
else:
    !7z x -y -p"$ZIP_PASSWORD" -o/content/ "$ZIP_PATH"
print('로컬 train.csv:', list(local_csv_files('train.csv')))

## 3. 라이브러리

In [ ]:
import gc, json, shutil, time
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

Image.MAX_IMAGE_PIXELS = None      # 큰 이미지 경고 해제
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# T4는 bf16을 지원하지 않으므로 fp16으로 떨어집니다. 공식 예시는 bf16입니다.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device:", DEVICE, "/ dtype:", DTYPE)
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0))

## 4. 파라미터

여기만 바꾸면 됩니다.

- `SPLIT` — `train`은 정답이 모두 있어 zero-shot 성능 측정이 가능합니다. `test`에는 정답 컬럼이 없습니다
- `MAX_NEW_TOKENS` — 이미지 해상도가 아니라 **글자량**이 좌우합니다 (픽셀수와의 상관 r=0.19).
  로컬 실측: 1024에서 잘림 약 25%. 1024→2048로 올려도 커버율이 늘지 않아(반복 루프가 원인) 1024를 권합니다
- `PROMPT` — `Spotting:`(텍스트+좌표) / `OCR:`(텍스트만, 토큰 약 절반).
  그 외 문장은 모델이 무시하고 Spotting으로 처리합니다
- `SYNC_EVERY` — 이 장수마다 Drive로 복사합니다. 런타임이 끊겨도 여기까지는 보존됩니다

In [ ]:
SPLIT = "train"             # train / test / dev
MAX_NEW_TOKENS = 1024
PROMPT = "Spotting:"
LIMIT = None               # 전체 처리. 정수를 넣으면 앞 N장만 (시험용)
SYNC_EVERY = 100           # Drive 저장 주기(장)

DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")   # 데이터 zip과 산출물이 있는 폴더

MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.6"
# 모델 1.8GB. Colab의 기본 HF 캐시는 세션과 함께 사라지므로
# Drive에 캐시하면 런타임을 새로 띄워도 다시 받지 않습니다.
CACHE_MODEL_ON_DRIVE = True
MODEL_CACHE = (DRIVE_BASE / "hf_cache") if CACHE_MODEL_ON_DRIVE else None
if MODEL_CACHE:
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("/content/data")   # 6번 셀에서 실제 위치로 다시 잡습니다   # 6번 셀에서 실제 위치로 다시 잡습니다
LOCAL_CACHE = Path("/content/ocr_cache")                       # 빠른 로컬 디스크
DRIVE_CACHE = DRIVE_BASE / "ocr_cache"                         # 영속 저장소
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

FILENAME = f"{SPLIT}_spotting.jsonl"
LOCAL_PATH, DRIVE_PATH = LOCAL_CACHE / FILENAME, DRIVE_CACHE / FILENAME
print(f"{SPLIT} / {PROMPT!r} / max_new_tokens={MAX_NEW_TOKENS} / limit={LIMIT}")
print("영속 저장:", DRIVE_PATH)
print("모델 캐시:", MODEL_CACHE or "세션 임시(재시작 시 1.8GB 재다운로드)")

BENCHMARK_SAMPLES = 16
BATCH_CANDIDATES = [1, 2, 4, 8]

## 5. OCR 엔진

로컬 `ocr_engine.py`와 같은 내용을 Colab용으로 인라인한 것입니다 (파일 의존성 없이 단독 실행되도록).

공식 근거: [모델 카드](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6)
- `task="spotting"` → 프롬프트 `Spotting:`
- 두 변이 모두 1500 미만이면 2배 확대(LANCZOS)
- `max_pixels = 2048*28*28` (spotting 전용 값)

In [ ]:
OCR_MAX_PIXELS = 2048 * 28 * 28      # 1,605,632 = 비전 토큰 2048개분
SPOTTING_UPSCALE_THRESHOLD = 1500

def spotting_image(image_path):
    """공식 spotting 전처리. 공통 축소 규칙을 적용하지 않습니다."""
    with Image.open(image_path) as im:
        im = im.convert("RGB")
    if im.width < SPOTTING_UPSCALE_THRESHOLD and im.height < SPOTTING_UPSCALE_THRESHOLD:
        im = im.resize((im.width * 2, im.height * 2), Image.Resampling.LANCZOS)
    return im

def _read_pixel_bounds(ip):
    """transformers 5.x 내장 클래스는 size가 SizeDict(shortest_edge/longest_edge = 픽셀 수),
    모델 폴더의 remote code 클래스는 min_pixels/max_pixels 속성을 씁니다."""
    size = getattr(ip, "size", None)
    def pick(*keys):
        for key in keys:
            v = size.get(key) if isinstance(size, dict) else getattr(size, key, None)
            if v:
                return int(v)
        return None
    low = pick("shortest_edge", "min_pixels") or getattr(ip, "min_pixels", None)
    high = pick("longest_edge", "max_pixels") or getattr(ip, "max_pixels", None)
    if not low or not high:
        raise ValueError(f"OCR 픽셀 설정을 찾지 못했습니다: {type(ip).__name__}, size={size}")
    return int(low), int(high)

def _probe_pixels(proc, edge=2048):
    grid = proc.image_processor(images=Image.new("RGB", (edge, edge)),
                               return_tensors="pt")["image_grid_thw"]
    return int(grid[0][1]) * int(grid[0][2]) * proc.image_processor.patch_size ** 2

def configure_ocr_pixels(proc, max_pixels=OCR_MAX_PIXELS):
    """픽셀 상한을 spotting 값으로 올리고 실측으로 검증합니다.

    apply_chat_template(images_kwargs=...)는 transformers 5.x에서 무시되므로
    프로세서 객체에 직접 설정합니다.
    """
    ip = proc.image_processor
    min_pixels, default_max = _read_pixel_bounds(ip)
    before = _probe_pixels(proc)
    size = getattr(ip, "size", None)
    if isinstance(size, dict):
        upd = dict(size)
        keys = ("shortest_edge", "longest_edge") if size.get("shortest_edge") else ("min_pixels", "max_pixels")
        upd[keys[0]], upd[keys[1]] = min_pixels, max_pixels
        ip.size = upd
    elif size is not None and getattr(size, "shortest_edge", None):
        size.shortest_edge, size.longest_edge = min_pixels, max_pixels
    if hasattr(ip, "min_pixels"):
        ip.min_pixels = min_pixels
    if hasattr(ip, "max_pixels"):
        ip.max_pixels = max_pixels
    after = _probe_pixels(proc)
    if not before < after <= max_pixels:
        raise RuntimeError(f"OCR 픽셀 상한 미적용: 2048px 입력이 {before} -> {after} "
                           f"(기대: {default_max} 초과, {max_pixels} 이하)")
    print(f"OCR 픽셀 범위 {min_pixels}~{max_pixels} 적용 확인 "
          f"(2048px 입력: {before} -> {after} 픽셀)")

def load_ocr():
    kw = {"cache_dir": str(MODEL_CACHE)} if MODEL_CACHE else {}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **kw)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=DTYPE, **kw).to(DEVICE).eval()
    proc.tokenizer.padding_side = "left"
    configure_ocr_pixels(proc)
    return model, proc

@torch.inference_mode()
def run_spotting(model, proc, image, max_new_tokens, prompt=PROMPT):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = proc.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    prefix_len = inputs["input_ids"].shape[-1]
    # 이 모델의 generation_config는 use_cache=False라서 매 스텝 전체를 다시 계산합니다.
    # greedy이므로 캐시를 켜도 생성 토큰은 동일하고 속도만 회복됩니다(로컬 실측 25배 차이).
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, use_cache=True)
    new = out[0, prefix_len:]
    eos = model.generation_config.eos_token_id
    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))
    truncated = len(new) >= max_new_tokens and (not len(new) or int(new[-1]) not in eos_ids)
    return proc.decode(new, skip_special_tokens=True).strip(), bool(truncated), int(len(new))

print("OCR 엔진 준비 완료")

@torch.inference_mode()
def run_batch(model, proc, rows):
    images = []
    try:
        torch.cuda.synchronize()
        start = time.perf_counter()
        images = [spotting_image(DATA_DIR / row['path']) for row in rows]
        messages = [[{'role': 'user', 'content': [
            {'type': 'image', 'image': im}, {'type': 'text', 'text': PROMPT}]}]
            for im in images]
        import inspect
        padding_kwargs = ({'processor_kwargs': {'padding': True}}
                          if 'processor_kwargs' in inspect.signature(proc.apply_chat_template).parameters
                          else {'padding': True})
        inputs = proc.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True, **padding_kwargs,
            return_dict=True, return_tensors='pt').to(model.device)
        prefix_len = inputs['input_ids'].shape[-1]
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                do_sample=False, use_cache=True)
        eos = model.generation_config.eos_token_id
        eos_ids = set([] if eos is None else ([eos] if isinstance(eos, int) else eos))
        results = []
        # Completed rows are padded until the longest row finishes. Strip at first EOS.
        for row, im, sequence in zip(rows, images, output[:, prefix_len:].tolist()):
            end = next((i + 1 for i, token in enumerate(sequence) if token in eos_ids), None)
            tokens = sequence if end is None else sequence[:end]
            results.append({
                'id': row['id'], 'path': row['path'],
                'ocr_raw': proc.decode(tokens, skip_special_tokens=True).strip(),
                'ocr_tokens': len(tokens),
                'ocr_truncated': end is None and len(tokens) >= MAX_NEW_TOKENS,
                'max_new_tokens': MAX_NEW_TOKENS, 'prompt': PROMPT,
                'image_width': im.width, 'image_height': im.height,
                'batch_size': len(rows), 'seconds_kind': 'batch_amortized',
            })
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        for result in results:
            result['seconds'] = elapsed / len(rows)
            result['batch_seconds'] = elapsed
        return results
    finally:
        for im in images:
            im.close()

def clear_oom():
    gc.collect()
    torch.cuda.empty_cache()

def signature(rows):
    return [(r['ocr_raw'], r['ocr_tokens'], r['ocr_truncated']) for r in rows]

def choose_batch_size(model, proc, todo):
    sample = todo.sample(n=min(BENCHMARK_SAMPLES, len(todo)), random_state=42).to_dict('records')
    print('벤치마크 워밍업 중...', flush=True)
    run_batch(model, proc, sample[:1])  # warm-up, excluded from timing
    reference, reports = None, []
    for batch_size in sorted(set([1] + BATCH_CANDIDATES)):
        if batch_size > len(sample):
            continue
        clear_oom()
        torch.cuda.reset_peak_memory_stats()
        outputs = []
        torch.cuda.synchronize()
        start = time.perf_counter()
        oom = False
        try:
            for offset in tqdm(range(0, len(sample), batch_size),
                               desc=f'벤치마크 batch={batch_size}', unit='batch'):
                outputs.extend(run_batch(model, proc, sample[offset:offset + batch_size]))
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:
            clear_oom()
            if batch_size == 1:
                raise RuntimeError('배치 1도 GPU 메모리가 부족합니다. 다른 모델을 해제하세요.')
            print(f'배치 {batch_size}: OOM — 제외')
            continue
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        if reference is None:
            reference = signature(outputs)
        exact = signature(outputs) == reference
        report = dict(batch_size=batch_size, seconds_per_image=elapsed / len(sample),
                      peak_gib=torch.cuda.max_memory_allocated() / 1024**3,
                      matches_single=exact)
        reports.append(report)
        print(report, flush=True)
    best = min((r for r in reports if r['matches_single']), key=lambda r: r['seconds_per_image'])
    report_path = DRIVE_CACHE / f'{SPLIT}_a100_benchmark.json'
    report_path.write_text(json.dumps({
        'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
        'transformers': __import__('transformers').__version__,
        'sample_ids': [r['id'] for r in sample], 'results': reports,
        'selected_batch_size': best['batch_size'],
        'model': MODEL_ID, 'prompt': PROMPT, 'max_new_tokens': MAX_NEW_TOKENS,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"선택: 배치 {best['batch_size']} / 샘플 기준 남은 작업 약 "
          f"{len(todo) * best['seconds_per_image'] / 3600:.2f}시간 (추정)")
    return best['batch_size']

## 6. 데이터 로드·검증
이미지가 실제로 존재하는지 먼저 확인합니다. 긴 작업 도중에 파일 누락을 발견하는 상황을 막습니다.

In [ ]:
def find_data_dir(split):
    """압축 해제 위치가 zip 구조에 따라 달라지므로 실제 위치를 찾습니다."""
    for base in (Path("/content/data"), Path("/content")):
        if (base / f"{split}.csv").is_file() and (base / split).is_dir():
            return base
    for csv in local_csv_files(f"{split}.csv"):
        if "drive" in csv.parts:          # Drive 원본은 제외
            continue
        if (csv.parent / split).is_dir():
            return csv.parent
    listing = sorted(x.name for x in Path("/content").iterdir())
    raise FileNotFoundError(
        f"{split}.csv 와 {split}/ 폴더를 /content 아래에서 찾지 못했습니다. "
        f"/content 내용: {listing}. "
        "2번 셀의 압축 해제가 성공했는지, ZIP_PATH와 비밀번호가 맞는지 확인하세요.")

DATA_DIR = find_data_dir(SPLIT)
print("데이터 위치:", DATA_DIR)

df = pd.read_csv(DATA_DIR / f"{SPLIT}.csv", keep_default_na=False)
assert df.id.is_unique and df.path.is_unique, "id 또는 path 중복"
missing = [p for p in df.path if not (DATA_DIR / p).is_file()]
assert not missing, f"이미지 누락 {len(missing)}개. 예: {missing[:3]}"
if LIMIT:
    df = df.head(LIMIT).copy()
print(f"{len(df)}장 확인 완료")

## 7. 이어하기 준비

로컬과 Drive 캐시를 모두 읽어 완료 ID를 합칩니다. 설정이나 동일 ID의 결과가 충돌하면 중단합니다.
파일 중간 손상은 중단하고, 끊긴 마지막 줄만 제외합니다.

In [ ]:
def read_cache(path):
    records = {}
    if not path.is_file():
        return records
    lines = path.read_text(encoding='utf-8').splitlines()
    for index, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            key = item['id']
        except (json.JSONDecodeError, KeyError):
            if index != len(lines) - 1:
                raise ValueError(f'캐시 중간 손상: {path}, 줄 {index + 1}')
            print('중단으로 손상된 마지막 줄 제외:', path)
            continue
        if item.get('max_new_tokens') != MAX_NEW_TOKENS or item.get('prompt') != PROMPT:
            raise ValueError(f'캐시 설정 불일치: {path}, {key}')
        if key in records and records[key] != item:
            raise ValueError(f'같은 ID의 서로 다른 결과: {key}')
        records[key] = item
    return records

# Read both before writing: resume also works when either local or Drive has more rows.
records = read_cache(DRIVE_PATH)
for key, item in read_cache(LOCAL_PATH).items():
    if key in records and records[key] != item:
        raise ValueError(f'로컬/Drive 결과 충돌: {key}. 원본을 확인하세요.')
    records[key] = item
if records:
    temp = LOCAL_PATH.with_suffix('.tmp')
    temp.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n'
                            for r in records.values()), encoding='utf-8')
    temp.replace(LOCAL_PATH)
done = set(records)
todo = df[~df.id.isin(done)]
print(f'전체 {len(df)}장 / 대상 중 완료 {len(df) - len(todo)}장 / 남음 {len(todo)}장')
del records

## 8. 벤치마크 → 전체 OCR

단일 처리와 샘플 결과가 일치하는 가장 빠른 배치를 선택합니다.
실행 중 메모리가 부족하면 해당 미완료 배치를 절반으로 줄여 재시도합니다.
Ctrl+C/중지 시 완료분을 Drive에 저장합니다. 강제 런타임 종료는 마지막 동기화 이후 결과를 잃을 수 있습니다.
배치 결과의 `seconds`는 개별 지연 시간이 아니라 **배치 시간 / 이미지 수**입니다.

In [ ]:
def sync_to_drive():
    if LOCAL_PATH.is_file():
        temp = DRIVE_PATH.with_suffix('.jsonl.tmp')
        shutil.copy(LOCAL_PATH, temp)
        temp.replace(DRIVE_PATH)

# Recompute on every execution, including when this cell is rerun after interruption.
done = set(read_cache(LOCAL_PATH))
todo = df[~df.id.isin(done)]
if todo.empty:
    print('이미 모두 완료되었습니다.')
else:
    model, proc = load_ocr()
    try:
        batch_size = choose_batch_size(model, proc, todo)
        torch.cuda.reset_peak_memory_stats()
        rows = todo.to_dict('records')
        offset, since_sync = 0, 0
        started = time.perf_counter()
        with LOCAL_PATH.open('a', encoding='utf-8') as log, tqdm(total=len(rows), desc=f'OCR {SPLIT}') as bar:
            while offset < len(rows):
                current = rows[offset:offset + batch_size]
                oom = False
                try:
                    results = run_batch(model, proc, current)
                except torch.cuda.OutOfMemoryError:
                    oom = True
                if oom:
                    clear_oom()
                    if batch_size == 1:
                        raise RuntimeError('배치 1에서 OOM. 완료분 저장 후 중단합니다.')
                    batch_size = max(1, batch_size // 2)
                    print('메모리 부족: 배치를 줄여 재시도:', batch_size)
                    continue
                for result in results:
                    log.write(json.dumps(result, ensure_ascii=False) + '\n')
                log.flush()
                offset += len(results)
                since_sync += len(results)
                bar.update(len(results))
                if since_sync >= SYNC_EVERY:
                    sync_to_drive()
                    since_sync = 0
                seconds_per_image = (time.perf_counter() - started) / offset
                bar.set_postfix(batch=batch_size, sec_per_image=round(seconds_per_image, 2),
                                remaining_h=round((len(rows) - offset) * seconds_per_image / 3600, 2))
        print(f'완료 {offset}장 / 본 처리 {(time.perf_counter() - started) / 3600:.2f}시간')
        print(f'GPU peak {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB')
    finally:
        try:
            sync_to_drive()
        finally:
            del model, proc
            clear_oom()
    print('Drive 저장:', DRIVE_PATH)

## 9. 캐시 점검
GPU가 필요 없습니다. 3·4·6번 셀만 실행한 뒤 이 셀로 바로 와도 됩니다.

In [ ]:
cache = pd.read_json(LOCAL_PATH if LOCAL_PATH.is_file() else DRIVE_PATH, lines=True)
print(f"캐시 {len(cache)}행 / 대상 {len(df)}행")
assert cache.id.is_unique, "캐시에 id 중복"
missing_ids = set(df.id) - set(cache.id)
print("누락:", len(missing_ids), "" if not missing_ids else list(missing_ids)[:3])

# 컬럼명 empty는 DataFrame.empty 속성과 충돌하므로 is_empty를 씁니다.
cache["is_empty"] = cache.ocr_raw.astype(str).str.strip() == ""
print(f"\n빈 결과       {cache['is_empty'].sum():5d}장 ({cache['is_empty'].mean():.1%})")
print(f"길이 제한 도달 {cache['ocr_truncated'].sum():5d}장 ({cache['ocr_truncated'].mean():.1%})")
print(f"\n생성 토큰 분포:\n{cache.ocr_tokens.describe().round(1).to_string()}")
print(f"\n장당 평균 {cache.seconds.mean():.2f}초")
if len(cache) < len(df):
    remain = (len(df) - len(cache)) * cache.seconds.mean() / 3600
    print(f"→ 남은 {len(df) - len(cache)}장 예상: 약 {remain:.1f}시간")
print("\n샘플 2건:")
for _, r in cache.head(2).iterrows():
    print(f"  {r.id} ({r.ocr_tokens}토큰): {str(r.ocr_raw)[:200]}")

## 이후 실험에서 읽는 법

```python
import pandas as pd
cache = pd.read_json("/content/drive/MyDrive/SSAFY/AI_Challenge2/ocr_cache/train_spotting.jsonl", lines=True)
evidence = dict(zip(cache.id, cache.ocr_raw))     # id -> OCR 원문

train = pd.read_csv("/content/data/train.csv", keep_default_na=False)
train["ocr"] = train.id.map(evidence)
```

**저장 형식(JSONL)** — 한 줄 = JSON 하나 = 이미지 한 장:

| 필드 | 내용 |
|---|---|
| `id`, `path` | 이미지 식별자 |
| `ocr_raw` | OCR 원문. Spotting이면 글자마다 `<\|LOC_n\|>` 좌표 토큰 8개가 붙습니다 |
| `ocr_tokens` | 생성 토큰 수 |
| `ocr_truncated` | 길이 제한 도달 여부 |
| `max_new_tokens`, `prompt` | 이 줄을 만든 설정 |
| `seconds` | 소요 시간 |
| `image_width/height` | 전처리 후 크기 |

`ocr_raw`에 원문을 그대로 두므로 좌표 제거나 재포맷 같은 후처리는 나중에 얼마든지 가능합니다.

```python
import re
plain = re.sub(r"<\|LOC_\d+\|>", "", ocr_raw)   # 좌표 제거
```